In [78]:
import pandas as pd 
import numpy as np

In [79]:
df = pd.read_csv("Crop_Data.csv")
df.head()

,Nitrogen,Phosphorus,Potassium,Temperature,Humidity,pH_Value,Rainfall,Crop
0,90,42,43,20.879744,82.002744,6.502985,202.935536,Rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,Rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,Rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,Rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,Rice


In [80]:
df.shape

(2200, 8)

In [82]:
X = df.drop("Crop", axis=1)
y = df["Crop"]
X.shape

(2200, 7)

In [83]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y =le.fit_transform(y)

In [84]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

y_train

array([16,  2, 10, ..., 12,  7, 10])

In [85]:
scaler = StandardScaler()

X_trained_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [86]:
# Now  work on the  ANN part 
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [87]:
X_train_tensor = torch.tensor(X_trained_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [88]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [107]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            
            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 22)
        )
        
    def forward(self, X):
        return self.model(X)
        
        

In [108]:
# Loss and optim
model = ANN()

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [109]:

epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        output =  model(xb)
        loss = criteria(output, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss/len(train_loader)

    print(f" epoch = {epoch+1}/{epochs}, loss = {train_loss}")


 epoch = 1/100, loss = 2.7943089962005616
 epoch = 2/100, loss = 1.8035505619916048
 epoch = 3/100, loss = 0.9123119982806119
 epoch = 4/100, loss = 0.5324842870235443
 epoch = 5/100, loss = 0.366461735421961
 epoch = 6/100, loss = 0.27527757693420757
 epoch = 7/100, loss = 0.21880620663816278
 epoch = 8/100, loss = 0.17931563041426918
 epoch = 9/100, loss = 0.15197012410922484
 epoch = 10/100, loss = 0.13156760836189443
 epoch = 11/100, loss = 0.11562616357749159
 epoch = 12/100, loss = 0.10191368121992458
 epoch = 13/100, loss = 0.09320773428136653
 epoch = 14/100, loss = 0.07967848930169236
 epoch = 15/100, loss = 0.07789908501912247
 epoch = 16/100, loss = 0.06790229921991175
 epoch = 17/100, loss = 0.06448199190199375
 epoch = 18/100, loss = 0.06244197355752641
 epoch = 19/100, loss = 0.054866871136155994
 epoch = 20/100, loss = 0.0504648663780906
 epoch = 21/100, loss = 0.04690217983654954
 epoch = 22/100, loss = 0.04460795812986114
 epoch = 23/100, loss = 0.041874539598145265
 e

In [ ]:
# For 1 Hidden Layer,  32 nodes loss = 0.030101546602831646
# For 1 Hidden Layer,  64 nodes loss = 0.02101434163156558
# For 2 Hidden Layers, 32 nodes, loss = 0.019440602365648374
# for 2 Hidden Layers, 64 nodes, loss = 0.007916331916666505

In [110]:
# Evaluate Phase
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        output = model(xb)
        
        # Yahan humne class ka index nikala aur 'predicted' mein save kiya
        _, predicted = torch.max(output, 1)

        # Fix 1: 'predict' ki jagah 'predicted' likha
        # Fix 2: yb ko yb.view(-1) kiya taaki size hamesha match ho
        correct += (predicted == yb.view(-1)).sum().item()
        
        total += yb.size(0)

# Final Accuracy Print
print(f"Accuracy: {(correct/total)*100:.2f}%")

Accuracy: 97.73%


In [ ]:
# For 1 Hidden Layer,  32 nodes Accuracy: 97.05%
# For 1 Hidden Layer,  64 nodes Accuracy: 96.82%
# For 2 Hidden Layers, 32 nodes Accuracy: 97.27%
# For 2 Hidden Layers, 64 nodes Accuracy: 97.73%